In [ ]:
from typing import Tuple
from pathlib import Path

import pandas as pd
import itables
from itables import show

from utils.data import load_leetcodedataset_data
from agents import Tutor, MyAgent
from langchain.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
%load_ext autoreload
%autoreload 2

itables.init_notebook_mode()
itables.options.maxBytes = 131072
itables.options.maxColumns = 0
itables.options.columnDefs=[{"width": "120px", "targets": "_all"}]

PATH_DATA = Path("./data/")

from dotenv import load_dotenv
import os

load_dotenv("secrets/openai.env")  
api_key = os.getenv("OPENAI_API_KEY")
# print("API Key:", api_key)

## Data

In [ ]:
df_train, df_test = load_leetcodedataset_data(PATH_DATA)

In [ ]:
df_train

In [ ]:
problem = df_train.iloc[0]

## Agents Test

In [ ]:
from prompts.tutor.baseline import CODING_PRACTICE_PROMPT
from pprint import pprint 

In [ ]:
tutor_system_message =  CODING_PRACTICE_PROMPT.format(
    problem_description = problem["problem_description"] ,
    starter_code = problem["starter_code"],
)
pprint(tutor_system_message) 

In [ ]:
problem

In [ ]:
pprint(problem.problem_description)

In [ ]:
print(problem.starter_code)

In [ ]:
pprint(problem.entry_point)

In [ ]:
problem.input_output

## Test Tutor Agent

In [ ]:
tutor_system_message

In [ ]:
tutor = Tutor(tutor_system_message)

In [ ]:
tutor.invoke_pprint(HumanMessage(content="Hello. Can you explain the problem to me?"))

In [ ]:
tutor.invoke_pprint(HumanMessage(content="What about using two for loops to solve this problem?"))

In [ ]:
# NOTE: The Tutor leaked the solution in the explanation!! 
tutor.invoke_pprint(HumanMessage(content="Yes, help me implement the approach."))

In [ ]:
tutor.invoke_pprint(HumanMessage(content="I think It would be better if I try something like a set or hashmap to solve this problem in O(n) time. "))

In [ ]:
s = """
What about? 
num_to_pos = {}
for idx,num in nums:
    num_to_pos[num] = idx

for num in nums:
    missing = target - num
    if missing in num_to_pos and missing != num:
        return [num_to_pos[num], num_to_pos[missing]]
"""
tutor.invoke_pprint(HumanMessage(content=s))

In [ ]:
s = """
Thank you for the feedback. What about? 
num_to_pos = {}
for idx,num in enumerate(nums):
    num_to_pos[num] = idx

for idx,num in enumerate(nums):
    missing = target - num
    if missing in num_to_pos and num_to_pos[missing] != idx:
        return [idx, num_to_pos[missing]]
"""
tutor.invoke_pprint(HumanMessage(content=s))

In [ ]:
s = """
nums = [2,7,11,15]
target = 9
num_to_pos = {}
for idx,num in enumerate(nums):
    num_to_pos[num] = idx

for idx,num in enumerate(nums):
    missing = target - num
    if missing in num_to_pos and num_to_pos[missing] != idx:
        print([idx, num_to_pos[missing]])
"""

def run_code(code:str)->str: 
    try: 
        out = exec(code)
        return out
    except Exception as e:
        return f"Error executing code: {e}"

out = run_code(s)
out

## Student Agent + Tutor Interaction

In [ ]:
from agents import MyAgent, Tutor, Student
from prompts.tutor.baseline import CODING_PRACTICE_PROMPT
from prompts.student.baseline import CODING_STUDENT_PROMPT

tutor_system_message =  CODING_PRACTICE_PROMPT.format(
    problem_description = problem["problem_description"] ,
    starter_code = problem["starter_code"],
)

student_system_message = CODING_STUDENT_PROMPT.format(
    problem_description = problem["problem_description"] ,
    starter_code = problem["starter_code"],
    programming_level = "beginner",
)

student_system_message

In [ ]:
tutor = Tutor(tutor_system_message)
student = Student(student_system_message)

In [ ]:
r = student.invoke_pprint(AIMessage(content="Start."))
r 

In [ ]:
from tqdm import tqdm
tutor = Tutor(tutor_system_message)
student = Student(student_system_message)


tutor_message = AIMessage(content="Start.")
interaction_messages = []

for i in tqdm(range(5)):
    student_message = student.invoke(tutor_message)
    interaction_messages.append(student_message)

    tutor_message = tutor.invoke(student_message)
    interaction_messages.append(tutor_message)

In [ ]:
for msg in interaction_messages:
    if isinstance(msg, HumanMessage):
        print("\nSTUDENT MESSAGE:")
    elif isinstance(msg, AIMessage):
        print("\nTUTOR MESSAGE")
    else: 
        raise Exception("Message weird class")

    pprint(msg.content)

## Running Student Code

In [ ]:
import json
student_message = interaction_messages[-6]
pprint(student_message)
student_message.content

student_dict = json.loads(student_message.content)
pprint(student_dict)
student_python_code = student_dict["python_code"]
pprint(student_python_code)

In [ ]:
from utils.code_dependencies import *
from utils.code_processing import get_code_definitions, show_all_dataset_definitions, code_runs

In [ ]:
# show_all_dataset_definitions(df_train)
# show_all_dataset_definitions(df_test)

In [ ]:
problem.starter_code

In [ ]:
code_definitions = get_code_definitions(problem.starter_code)
exec(code_definitions)
if code_runs(student_python_code):
    exec(student_python_code)
else: 
    raise Warning("Problem running student's code")

exec(student_python_code)
# exec(student_python_code, {'List': List})

In [ ]:
Solution.twoSum

In [ ]:
print(type(problem.input_output))
problem.input_output[:5]

## Check Outputs

- Are all outputs primitives or data structures comparable with '=='?: No. Ex: problem1 output Optional[ListNode]
- Can we run the test instead?: Probably not. See problem1, test depends on function is_same_list
- Where can I get that function?: IDK lol 

In [ ]:
problem.entry_point

In [ ]:
problem.input_output

In [ ]:
set([1,2])==set([2,1]), [1,2]==[2,1]

In [ ]:
output_types = set()
different_prompts = set()
for idx,p in enumerate(df_train.iloc):
    # print(p.starter_code)
    output = p.starter_code.split('->')[-1]
    if output not in output_types:
        output_types.add(output)
        print(idx)
        print(p.starter_code)
# print(output_types)

In [ ]:
problem1 = df_train.iloc[500]
print(problem1.test)

In [ ]:
exec(problem1.prompt)
help(is_same_list)

In [ ]:
## Print different prompts (dependency imports)
# different_prompts = set(p.prompt for p in df_train.iloc)
# different_prompts.update(p.prompt for p in df_test.iloc )
# for prompt in different_prompts:
#     print(prompt)


In [ ]:
print("problem.entry_point:", problem.entry_point)
eval(f"{problem.entry_point}(nums = [3, 3],target = 6) == [0, 1]")

In [ ]:
pprint(problem.test)

In [ ]:
from typing import Literal
def numeric_test_score(problem, verbose:Literal[0,1,2]=0):
    asserts = problem.test.split("assert")[1:]
    asserts = [ass.strip() for ass in asserts]
    if verbose>0: print(asserts)
    
    count_passed = 0
    for idx,ass in enumerate(asserts):
        try:
            # replace first occurrence of substring by entry point
            assestent_line_code = ass.replace("candidate", problem.entry_point, 1) 
            # evaluate assert
            evaluation_passed = eval(assestent_line_code)
            if (verbose == 2) or (verbose == 1 and not evaluation_passed):
                print(idx, assestent_line_code)
                print(evaluation_passed, '\n')
            if evaluation_passed:
                count_passed += 1
        except: pass
    # return proportion of passed asserts 
    return count_passed / len(asserts)

numeric_test_score(problem, verbose=1)

In [ ]:
# NOTE: Expected Solution().twoSum(nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],target = 17) == [7, 8]
# which is also a valid solution! 
Solution().twoSum(nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],target = 17)

In [ ]:
print(student_python_code)

## Test Judges

In [ ]:
from prompts.judge.baseline import STUDENT_JUDGE_PROMPT, TUTOR_JUDGE_PROMPT

student_judge_system_message =  STUDENT_JUDGE_PROMPT.format(
    problem_description = problem["problem_description"] ,
    starter_code = problem["starter_code"],
)

tutor_judge_system_message = TUTOR_JUDGE_PROMPT.format(
    problem_description = problem["problem_description"] ,
    starter_code = problem["starter_code"],
)

# pprint(tutor_judge_system_message)

from agents import StudentJudge, TutorJudge

student_judge_agent = StudentJudge(system_message=student_judge_system_message)
tutor_judge_agent = TutorJudge(system_message=tutor_judge_system_message)

In [ ]:
for idx,message in enumerate(interaction_messages):
    if isinstance(message,HumanMessage):
        print("STUDENT:")
        print(message)
        student_judgment = student_judge_agent.invoke(message) # 23224 -> 44444
        pprint(json.loads(student_judgment.content))
    elif isinstance(message,AIMessage):
        print("TUTOR:")
        print(message)
        tutor_judgment = tutor_judge_agent.invoke(message)
        pprint(json.loads(tutor_judgment.content))
        print(40*"##")
    else: raise Exception("Unkown message type")

## All Agents

In [ ]:
import json
from collections import OrderedDict

from tqdm import tqdm
import numpy as np 
from typing import Literal

from agents import Tutor, Student
from agents import StudentJudge, TutorJudge
from prompts.tutor.baseline import CODING_PRACTICE_PROMPT, TUTOR_PEDAGOGICAL_MOVES_PROMPT
from prompts.student.baseline import CODING_STUDENT_PROMPT
from prompts.judge.baseline import STUDENT_JUDGE_PROMPT, TUTOR_JUDGE_PROMPT
from utils.code_processing import code_runs, get_code_definitions, numeric_test_score

def load_formatted_prompts(
        problem, 
        student_programming_level: Literal["beginner", "intermediate", "advanced"]="beginner"
    )->None:
    tutor_system_message =  CODING_PRACTICE_PROMPT.format(
        problem_description = problem["problem_description"] ,
        starter_code = problem["starter_code"],
    )

    student_system_message = CODING_STUDENT_PROMPT.format(
        problem_description = problem["problem_description"] ,
        starter_code = problem["starter_code"],
        programming_level = student_programming_level,
    )

    tutor_judge_system_message = TUTOR_JUDGE_PROMPT.format(
        problem_description = problem["problem_description"] ,
        starter_code = problem["starter_code"],
    )

    student_judge_system_message =  STUDENT_JUDGE_PROMPT.format(
        problem_description = problem["problem_description"] ,
        starter_code = problem["starter_code"],
    )
    return (
        tutor_system_message,
        student_system_message,
        tutor_judge_system_message,
        student_judge_system_message,
    )


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

ACTIONS = ["SOCRATIC_PROBE", "CONCEPTUAL_HINT", "STRUCTURAL_SCAFFOLD"]

STATE_COLS = ["i", "last_student_level", "last_tutor_level", "last_reward", "last_action"] #,"idx"]
MODEL_COLS = STATE_COLS + ["action"]
TARGET_COL = "reward"

memory = []          # list of transition dicts
reward_model = None  # trained sklearn pipeline

# ----------------------------
# Data logging
# ----------------------------
def log_transition(
    idx: int,
    i: int,
    last_student_level: int,
    last_tutor_level: int,
    last_action: str,
    last_reward: float,
    action: str,
    reward: float,
):
    memory.append(
        {
            # "idx": idx,
            "i": i,
            "last_student_level": last_student_level,
            "last_tutor_level": last_tutor_level,
            "last_action": str(last_action),
            "last_reward": float(last_reward),
            "action": str(action),
            "reward": float(reward),
        }
    )

def get_memory_df() -> pd.DataFrame:
    if not memory:
        return pd.DataFrame(columns=MODEL_COLS + [TARGET_COL])
    return pd.DataFrame(memory)

# ----------------------------
# Reward model: (state, action) -> expected reward
# ----------------------------
def train_reward_model(df: pd.DataFrame):
    pre = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["last_action", "action"]),
            ("num", "passthrough", ["i", "last_student_level", "last_tutor_level", "last_reward"]), 
        ]
    )
    model = Pipeline(
        steps=[
            ("pre", pre),
            ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ]
    )

    X = df[MODEL_COLS]
    y = df[TARGET_COL].astype(float)
    model.fit(X, y)
    return model

def maybe_retrain(min_rows: int = 200, retrain_every: int = 50):
    global reward_model
    n = len(memory)
    if n < min_rows:
        return
    if n % retrain_every != 0:
        return

    df = get_memory_df()
    # Need some reward variation for meaningful fit
    if df[TARGET_COL].nunique() < 2:
        return
    print("- - - -RETRAIN MODEL.")
    reward_model = train_reward_model(df)

# ----------------------------
# Action selection
# ----------------------------
def _state_dict(
    idx: int,
    i: int,
    last_student_level: int,
    last_tutor_level: int,
    last_reward: float,
    last_action: str,
):
    return {
        # "idx": idx,
        "i": i,
        "last_student_level": last_student_level,
        "last_tutor_level": last_tutor_level,
        "last_reward": float(last_reward),
        "last_action": str(last_action),
    }


def _softmax(x: np.ndarray, temp: float = 1.0) -> np.ndarray:
    t = max(temp, 1e-6)
    z = x / t
    z = z - np.max(z)
    p = np.exp(z)
    p = p / p.sum()
    return p

def get_next_action(
    idx: int,
    i: int,
    last_student_level: int,
    last_tutor_level: int,
    last_reward: float = 0.0,
    last_action: str = "NONE",
    eps: float = 0.10,      # exploration probability
    temp: float = 0.7,      # lower = greedier, higher = more random
) -> str:
    global reward_model

    # Cold-start or epsilon exploration
    if (idx==0 or i==0) or (reward_model is None) or (np.random.rand() < eps):
        return str(np.random.choice(ACTIONS))

    s = _state_dict(idx, i, last_student_level, last_tutor_level, last_reward, last_action)
    candidates = pd.DataFrame([{**s, "action": a} for a in ACTIONS])[MODEL_COLS]
    pred_rewards = reward_model.predict(candidates)  # E[r | s,a]

    probs = _softmax(pred_rewards, temp=temp)
    print(10*"->")
    print("ACTIONS:", ACTIONS)
    print("pred_rewards:", pred_rewards)
    print("pred_probs:", probs)
    return str(np.random.choice(ACTIONS, p=probs))

In [ ]:
all_interaction_messages = []
for idx, problem in enumerate(df_train[0:4].iloc):
    # Load prompts
    tutor_system_message, student_system_message, tutor_judge_system_message, student_judge_system_message = load_formatted_prompts(problem, "beginner")

    # Instance agents
    tutor_agent = Tutor(tutor_system_message)
    student_agent = Student(student_system_message)
    student_judge_agent = StudentJudge(system_message=student_judge_system_message)
    tutor_judge_agent = TutorJudge(system_message=tutor_judge_system_message)

    # Run initial required code
    code_definitions = get_code_definitions(problem.starter_code)

    interaction_messages = []
    
    # Initialize state variables (no prior interaction)
    last_student_level, last_tutor_level, last_reward, last_action = (-1, -1, -1.0, "NONE")
    
    # Keep track of student's last message for context
    student_message = HumanMessage(content="I'm starting to work on this problem.")

    for i in tqdm(range(10)):
        # Select pedagogical action (random for i==0, otherwise from reward model)
        action = get_next_action(idx, i, last_student_level, last_tutor_level, last_reward, last_action)
        print("Action selected:", action)
        
        # Update tutor's system message with the selected pedagogical move
        tutor_system_message = TUTOR_PEDAGOGICAL_MOVES_PROMPT.format(
            pedagogical_move=action,
            problem_description=problem["problem_description"],
            starter_code=problem["starter_code"],
        )
        tutor_agent.update_system_message(tutor_system_message)
        
        try:
            # Tutor responds based on the selected action
            tutor_message = tutor_agent.invoke(student_message)
            tutor_judgment = tutor_judge_agent.invoke(tutor_message)
            
            # Student responds to THIS tutor message (influenced by the action)
            student_message = student_agent.invoke(tutor_message)
            student_judgment = student_judge_agent.invoke(student_message)
        except Exception as e:
            log = {"stop": {"error": str(e)}}
            interaction_messages.append(log)
            break

        # Store interaction
        interaction_messages.append(OrderedDict({
            "tutor_message": tutor_message,
            "student_message": student_message,
            "tutor_judgment": tutor_judgment,
            "student_judgment": student_judgment,
        }))
        
        # Extract and evaluate student code
        student_message_dict = json.loads(student_message.content)
        student_python_code = student_message_dict["python_code"]

        code_runs_ = code_runs(code_definitions, student_python_code)
        if not code_runs_:
            print(f"Problem {idx}, step {i}: No running code or code does not run")
            coding_score = 0
        else:
            coding_score = numeric_test_score(problem, code_definitions, student_python_code, verbose=0)

        # Reward function. TODO: Divide by N??
        # Compute reward based on student's response to this tutor action
        student_judgment_dict = json.loads(student_judgment.content)
        tutor_judgment_dict = json.loads(tutor_judgment.content)
        
        CODE_RUNS_REWARD = 0.1
        LAMBDA = 0.3
        
        scaffold = 2 - abs(student_judgment_dict["student_level"] - tutor_judgment_dict["tutor_level"])
        if tutor_judgment_dict["leakage_detected"]:
            pedagogical_quality = -1
        else:
            pedagogical_quality = scaffold

        student_success = coding_score # TODO: change it by improvement rather than raw value?
        reward = (1-LAMBDA)*student_success + LAMBDA*pedagogical_quality + code_runs_*CODE_RUNS_REWARD
        print(f"reward:{reward}, (student_success:{student_success}, pedagogical_quality:{pedagogical_quality})")

        # Log transition for RL model
        log_transition(
            idx=idx,
            i=i,
            last_student_level=last_student_level,
            last_tutor_level=last_tutor_level,
            last_action=last_action,
            last_reward=last_reward,
            action=action,
            reward=reward,
        )

        # Retrain reward model periodically
        maybe_retrain(min_rows=10, retrain_every=3)

        # Check stop conditions
        if code_runs_ and coding_score == 1:
            log = {"stop": {"finished": "score equals 1"}}
            interaction_messages.append(log)
            break

        # Update state for next iteration
        last_student_level = student_judgment_dict["student_level"]
        last_tutor_level = tutor_judgment_dict["tutor_level"]
        last_reward = reward
        last_action = action

    # Finished interactions. 
    if "stop" not in interaction_messages[-1]:
        log = {"stop": {"problem not finished": "ran out of max number of iterations."}}
        interaction_messages.append(log)
    
    all_interaction_messages.append(interaction_messages)

In [ ]:
# Print first student solutions
from pprint import pprint 
for messages in all_interaction_messages[3]:
    print(40*"##")
    if "tutor_message" in messages:
        pprint("Tutor:")
        print(messages["tutor_message"])
    if "stop" not in messages:
        student_message_dict = json.loads(messages["student_message"].content)
        print("\nconversation:")
        pprint(student_message_dict["conversation"])
        print("\ncode:")
        pprint(student_message_dict["python_code"])
    else: 
        print(messages)

In [ ]:
print(all_interaction_messages[0])

In [ ]:
problem.entry_point

In [ ]:
problem.problem_description

In [ ]:
get_memory_df()

In [ ]:
df = get_memory_df()
print(df.reward.describe())
print(df.groupby('action').reward.mean())